# QC: friends seasons as a temporal-stability control

**Tier-3 robustness, not the primary analysis** (CLAUDE.md, "Respect the
analysis hierarchy" and "Exploratory QC, kept deliberately separate").
`friends` is the most task-homogeneous dataset available — every run is a
Friends episode — and its six seasons (s01-s06) were acquired in order across
most of the project. Splitting session pairs by same-/different-**season**
instead of same-/different-dataset isolates drift (scanner, subject state,
elapsed time) from cognitive context, which `qc_similarity.ipynb`'s
dataset split cannot: dataset and time are confounded there.

Season is perfectly confounded with elapsed time by design — that is the
point, not a flaw — but it also means within-season pairs are temporally
*adjacent* pairs. The lag curves below (similarity vs. season lag, vs. session
gap) are what separate "stable, with a task-free season label" from "smooth
drift"; the four-bin split alone cannot. Session number is the only available
time axis (no acquisition dates are stored in the timeseries files or the
index), so "elapsed time" means session ordinal, not calendar time. The lag
curves plot **both** within- and between-subject pairs on the same axes: if
between-subject similarity also drifts with lag, that points at a shared
(e.g. scanner-wide) drift rather than something specific to how a single
subject's data changes over the project.

This has no usable-data gate (the FD / duration criteria are still open —
CLAUDE.md, "Still open"). Deliberately kept out of the montage figure and out
of `run-group-stats`.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
source_dir = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

# invoke.yaml is not exposed as an env var by run-notebooks, so read the
# project root config directly for the parcellation and network order. The
# same root also holds analysis/, which is not on sys.path when nbconvert
# launches the kernel from an arbitrary cwd, so add it before importing.
project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.connectome_store import load_index, load_measure  # noqa: E402
from analysis.friends_seasons import attach_seasons, season_index, session_seasons  # noqa: E402
from analysis.similarity import (  # noqa: E402
    collect_pair_values,
    common_edge_mask,
    discover_connectome_files,
    fisher_z,
    pair_bin_labels,
    pair_bins,
    pair_frame,
    similarity_matrix,
    summarize_bins,
)

# Not a montage panel, so there is no panel_size box to render into — this
# notebook picks its own free figure size instead.
figure_dir = figures_base / "qc_friends_seasons"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURES = invoke_config.get("connectome_measures", ["pearson", "partial_ledoitwolf"])

connectome_dir = output_dir / "connectomes"
paths, skipped = discover_connectome_files(connectome_dir, PARCELLATION)
for path, reason in skipped:
    print(f"⚠️  skipping {path.name}: {reason}")
friends_paths = [p for p in paths if p.stem.startswith("friends_")]
if len(friends_paths) != 1:
    raise RuntimeError(f"expected exactly one friends connectome file, found {friends_paths}")
friends_path = friends_paths[0]
print(f"📂 using {friends_path.name}")

⚠️  skipping movie10_schaefer1000.h5: parcellation='schaefer1000'
📂 using friends_cneuromod2026.h5


In [2]:
# Season is not stored in the connectome index (multi-run sessions collapse
# `task` to "multi" — see CLAUDE.md), so it is re-derived here from the source
# h5 key names alone (cheap — no timeseries loaded), and attached once: every
# network/measure below shares the same set of sessions and the same drops.
full_index = load_index(friends_path)
full_index["_row"] = np.arange(len(full_index))

season_frame = session_seasons(source_dir / "cneuromod.all", PARCELLATION)
filtered_index, dropped = attach_seasons(full_index, season_frame)
row_indices = filtered_index["_row"].to_numpy()

print(f"📊 {len(full_index)} friends sessions -> {len(filtered_index)} kept "
      f"({dropped['boundary']} season-boundary, {dropped['unmatched']} unmatched dropped)")
filtered_index["season"].value_counts().sort_index()

📊 335 friends sessions -> 326 kept (9 season-boundary, 0 unmatched dropped)


season
s01    52
s02    51
s03    51
s04    50
s05    58
s06    64
Name: count, dtype: int64

In [3]:
# Colors keyed by bin label, shared across every panel and both measures so
# the four distributions stay visually comparable figure to figure. This
# dict is season-specific — the dataset-keyed one in qc_similarity.ipynb
# cannot be reused since the label strings differ.
SEASON_BINS = pair_bin_labels("season")
BIN_COLORS = {
    SEASON_BINS[0]: "#1b9e77",
    SEASON_BINS[1]: "#7570b3",
    SEASON_BINS[2]: "#d95f02",
    SEASON_BINS[3]: "#999999",
}

# Lag-curve styling: color encodes within- vs. between-subject (matching the
# bin-plot palette above — teal for within-subject, orange for between), and
# marker/linestyle encodes which time axis (season lag vs. binned session gap).
LAG_STYLE = {
    "within-subject": {"color": "#1b9e77", "season": ("o", "-"), "session": ("s", "--")},
    "between-subject": {"color": "#d95f02", "season": ("^", "-"), "session": ("D", "--")},
}

bin_summary_rows = []
lag_summary_rows = []


def _lag_stats(frame, group_col):
    stats = frame.groupby(group_col)["similarity"].agg(
        n="count", median="median",
        q25=lambda s: np.percentile(s, 25), q75=lambda s: np.percentile(s, 75),
    ).reset_index()
    return stats


for measure in MEASURES:
    bins_fig, bins_axes = plt.subplots(3, 3, figsize=(12, 10), layout="constrained")
    bins_fig.suptitle(f"Friends session-pair similarity by season — {measure}")

    lag_fig, lag_axes = plt.subplots(3, 3, figsize=(12, 10), layout="constrained")
    lag_fig.suptitle(f"Friends session-pair similarity vs. temporal lag — {measure}")

    for network, bins_ax, lag_ax in zip(NETWORK_ORDER, bins_axes.flat, lag_axes.flat):
        array = load_measure(friends_path, measure, network)
        matrix = array[row_indices]
        z_matrix = fisher_z(matrix)
        valid = common_edge_mask(z_matrix)
        similarity = similarity_matrix(z_matrix)

        # Bin split: same-/different-subject x same-/different-season.
        bins = pair_bins(filtered_index, group_column="season", group_name="season")
        values_by_bin = collect_pair_values(similarity, bins, bin_labels=SEASON_BINS)
        summary = summarize_bins(values_by_bin, bin_labels=SEASON_BINS)
        summary["measure"] = measure
        summary["network"] = network
        summary["n_edges_valid"] = int(valid.sum())
        summary["n_edges_total"] = int(valid.size)
        bin_summary_rows.append(summary)

        all_values = np.concatenate([v for v in values_by_bin.values() if len(v)])
        bin_edges = (np.linspace(all_values.min(), all_values.max(), 40)
                     if len(all_values) else np.linspace(-1, 1, 40))
        for bin_label, values in values_by_bin.items():
            if len(values) == 0:
                continue
            counts, edges = np.histogram(values, bins=bin_edges, density=True)
            bins_ax.stairs(counts, edges, color=BIN_COLORS[bin_label], label=bin_label)
        bins_ax.set_title(network, fontsize=10)
        bins_ax.set_yticks([])
        bins_ax.annotate(f"edges {valid.sum()}/{valid.size}", xy=(0.02, 0.95),
                          xycoords="axes fraction", fontsize=7, va="top", color="0.4")

        # Lag curves, within- and between-subject alike: similarity vs. season
        # lag (0-5 seasons apart) and vs. binned session gap (session ordinal,
        # the only time axis available — no acquisition dates are stored).
        # Plotting both subject-relations on one axis is what makes the
        # within/between separation and each curve's own drift visible at once.
        pairs = pair_frame(similarity, filtered_index, columns=("subject", "season", "session"))
        pairs["season_lag"] = (
            pairs["season_i"].map(season_index) - pairs["season_j"].map(season_index)
        ).abs()
        pairs["session_gap"] = (
            pairs["session_i"].astype(int) - pairs["session_j"].astype(int)
        ).abs()

        for pair_type, group in (
            ("within-subject", pairs[pairs["subject_i"] == pairs["subject_j"]]),
            ("between-subject", pairs[pairs["subject_i"] != pairs["subject_j"]]),
        ):
            style = LAG_STYLE[pair_type]

            season_stats = _lag_stats(group, "season_lag")
            season_stats["measure"] = measure
            season_stats["network"] = network
            season_stats["pair_type"] = pair_type
            lag_summary_rows.append(season_stats)

            marker, linestyle = style["season"]
            lag_ax.plot(season_stats["season_lag"], season_stats["median"],
                        marker=marker, linestyle=linestyle, color=style["color"],
                        label=f"{pair_type}, season lag")
            lag_ax.fill_between(season_stats["season_lag"], season_stats["q25"],
                                 season_stats["q75"], color=style["color"], alpha=0.15)

            session_gap_bin = pd.qcut(group["session_gap"], q=6, duplicates="drop")
            session_stats = group.groupby(session_gap_bin, observed=True)["similarity"].agg(
                median="median", q25=lambda s: np.percentile(s, 25),
                q75=lambda s: np.percentile(s, 75),
            ).reset_index(drop=True)
            gap_x = np.linspace(0, season_stats["season_lag"].max() or 1, len(session_stats))

            marker, linestyle = style["session"]
            lag_ax.plot(gap_x, session_stats["median"], marker=marker, linestyle=linestyle,
                        color=style["color"], alpha=0.6, label=f"{pair_type}, session gap")
            lag_ax.fill_between(gap_x, session_stats["q25"], session_stats["q75"],
                                 color=style["color"], alpha=0.08)

        lag_ax.set_title(network, fontsize=10)
        lag_ax.set_xlabel("lag (season steps, or session-gap decile)", fontsize=7)
        lag_ax.tick_params(labelsize=7)

    bins_axes.flat[0].legend(fontsize=7, loc="upper right")
    bins_out_path = figure_dir / f"{measure}_season_bins.png"
    bins_fig.savefig(bins_out_path, dpi=150)
    plt.close(bins_fig)
    print(f"✅ wrote {bins_out_path}")

    lag_axes.flat[0].legend(fontsize=6, loc="center right")
    lag_out_path = figure_dir / f"{measure}_season_lag.png"
    lag_fig.savefig(lag_out_path, dpi=150)
    plt.close(lag_fig)
    print(f"✅ wrote {lag_out_path}")

✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/pearson_season_bins.png


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/pearson_season_lag.png


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/partial_ledoitwolf_season_bins.png


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/partial_ledoitwolf_season_lag.png


In [4]:
bin_summary_frame = pd.concat(bin_summary_rows, ignore_index=True)
bin_summary_frame = bin_summary_frame[
    ["measure", "network", "bin", "n", "median", "q25", "q75", "mean", "sd",
     "n_edges_valid", "n_edges_total"]
]
bin_summary_path = figure_dir / "season_pair_summary.tsv"
bin_summary_frame.to_csv(bin_summary_path, sep="\t", index=False)
print(f"✅ wrote {bin_summary_path}")

lag_summary_frame = pd.concat(lag_summary_rows, ignore_index=True)
lag_summary_frame = lag_summary_frame[
    ["measure", "network", "pair_type", "season_lag", "n", "median", "q25", "q75"]
]
lag_summary_path = figure_dir / "season_lag_summary.tsv"
lag_summary_frame.to_csv(lag_summary_path, sep="\t", index=False)
print(f"✅ wrote {lag_summary_path}")

bin_summary_frame.head(12)

✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/season_pair_summary.tsv
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/qc_friends_seasons/season_lag_summary.tsv


,measure,network,bin,n,median,q25,q75,mean,sd,n_edges_valid,n_edges_total
0,pearson,Vis,within-subject / within-season,1522,0.953068,0.941674,0.961594,0.950387,0.016459,13041,13041
1,pearson,Vis,within-subject / between-season,7607,0.943875,0.931278,0.953843,0.940837,0.018845,13041,13041
2,pearson,Vis,between-subject / within-season,7248,0.691274,0.665762,0.724349,0.694451,0.039671,13041,13041
3,pearson,Vis,between-subject / between-season,36598,0.691551,0.665399,0.722730,0.693395,0.038950,13041,13041
4,pearson,SomMot,within-subject / within-season,1522,0.916193,0.892915,0.935105,0.909411,0.036505,18721,18721
5,pearson,SomMot,within-subject / between-season,7607,0.904815,0.878988,0.922447,0.897083,0.036689,18721,18721
6,pearson,SomMot,between-subject / within-season,7248,0.609778,0.566799,0.648915,0.606597,0.059470,18721,18721
7,pearson,SomMot,between-subject / between-season,36598,0.609222,0.566025,0.648287,0.605727,0.059340,18721,18721
8,pearson,DorsAttn,within-subject / within-season,1522,0.946514,0.929035,0.958400,0.941638,0.022434,7381,7381
9,pearson,DorsAttn,within-subject / between-season,7607,0.938181,0.921182,0.951967,0.934263,0.024218,7381,7381
